# 02 - Feature engineering and selection

Engineer derived features, evaluate predictive power, and select the final feature set for modeling.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from backlink_pricing_model.core.notebook import init_notebook
from backlink_pricing_model.preprocessing.data_imputation import (
    impute_metrics_by_domain,
    summarize_imputation,
)
from backlink_pricing_model.preprocessing.data_loading import save_processed
from backlink_pricing_model.preprocessing.feature_engineering import (
    add_price_ratio,
    add_temporal_features,
)
from backlink_pricing_model.visualization.importance_plots import (
    plot_correlation_heatmap,
)

init_notebook()

## 1. Load cleaned data from notebook 01

In [ ]:
from backlink_pricing_model.core.environment import get_project_root

df = pd.read_parquet(get_project_root() / "data" / "processed" / "backlinks_cleaned.parquet")
print(f"Loaded {len(df):,} rows")
df.head()

## 2. Impute missing CF/TF using domain-level medians

In [ ]:
df_before = df.copy()
df = impute_metrics_by_domain(df)
summarize_imputation(df_before, df)

## 3. Engineer additional features

In [ ]:
# Add negotiation ratio and temporal features.
df = add_price_ratio(df)
df = add_temporal_features(df)

print(f"Columns after feature engineering: {list(df.columns)}")
df.describe()

## 4. Correlation analysis

In [ ]:
numeric_cols = ["final_price", "dr", "cf", "tf", "domain_traffic", "log_price", "log_traffic"]
corr_matrix = df[numeric_cols].corr()
plot_correlation_heatmap(corr_matrix, title="Numeric feature correlations").show()

## 5. Encode categorical features

In [ ]:
# Label-encode categorical features for tree-based models.
categorical_cols = ["tld", "country"]
label_encoders: dict[str, LabelEncoder] = {}

for col in categorical_cols:
    le = LabelEncoder()
    # Fill NaN with "unknown" before encoding.
    df[f"{col}_encoded"] = le.fit_transform(df[col].fillna("unknown"))
    label_encoders[col] = le
    n_classes = len(le.classes_)
    print(f"{col}: {n_classes} unique values")

## 6. Define final feature set and train/test split

In [ ]:
# Final feature columns for modeling.
FEATURE_COLS = [
    "dr",
    "cf",
    "tf",
    "log_traffic",
    "tld_encoded",
    "country_encoded",
    "year",
    "month",
    "quarter",
]
TARGET = "log_price"

# Drop rows with NaN in any feature or target column.
df_model = df.dropna(subset=[*FEATURE_COLS, TARGET])
print(f"Modeling dataset: {len(df_model):,} rows, {len(FEATURE_COLS)} features")

X = df_model[FEATURE_COLS]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

## 7. Save engineered datasets

In [ ]:
# Save full engineered dataset.
save_processed(df, "backlinks_engineered", subdir="engineered")

# Save train/test splits with target.
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

save_processed(train_df, "train_df", subdir="engineered")
save_processed(test_df, "test_df", subdir="engineered")

print("Saved engineered datasets to data/engineered/")